In [1]:
import mediapipe as mp
import cv2
import time

In [8]:

mp_drawing=mp.solutions.drawing_utils
mp_holistic=mp.solutions.holistic

face_draw_spec=mp_drawing.DrawingSpec(thickness=1,circle_radius=1,color=(255,180,0))
face_connect_spec=mp_drawing.DrawingSpec(thickness=1,circle_radius=1,color=(255,100,0))

right_draw_spec=mp_drawing.DrawingSpec(thickness=2,circle_radius=3,color=(0,220,255))
right_connect_spec=mp_drawing.DrawingSpec(thickness=2,circle_radius=2,color=(0,150,255))

left_draw_spec=mp_drawing.DrawingSpec(thickness=2,circle_radius=3,color=(255,80,180))
left_connect_spec=mp_drawing.DrawingSpec(thickness=2,circle_radius=2,color=(220,40,140))

pose_draw_spec=mp_drawing.DrawingSpec(thickness=3,circle_radius=3,color=(80,255,120))
pose_connect_spec=mp_drawing.DrawingSpec(thickness=3,circle_radius=2,color=(40,200,80))

cap=cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,2020)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT,1200)
cap.set(cv2.CAP_PROP_FPS,30)

frame_counter=0
prev_time=time.time()
fps=0

with mp_holistic.Holistic(min_detection_confidence=0.5,min_tracking_confidence=0.5,model_complexity=1) as holistic:

    while cap.isOpened():

        ret,frame=cap.read()

        if not ret:
            break

        frame_counter+=1

        if frame_counter%2!=0:
            continue

        frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        frame.flags.writeable=False
        results=holistic.process(frame)
        frame.flags.writeable=True
        frame=cv2.cvtColor(frame,cv2.COLOR_RGB2BGR)

        h,w,c=frame.shape

        if results.face_landmarks:
            mp_drawing.draw_landmarks(frame,results.face_landmarks,mp_holistic.FACEMESH_TESSELATION,face_draw_spec,face_connect_spec)
            x_list=[lm.x*w for lm in results.face_landmarks.landmark]
            y_list=[lm.y*h for lm in results.face_landmarks.landmark]
            x_min=max(0,int(min(x_list))-10)
            x_max=min(w,int(max(x_list))+10)
            y_min=max(0,int(min(y_list))-10)
            y_max=min(h,int(max(y_list))+10)
            cv2.rectangle(frame,(x_min,y_min),(x_max,y_max),(255,180,0),2)
            cv2.putText(frame,"FACE",(x_min,y_min-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,180,0),2)

        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(frame,results.left_hand_landmarks,mp_holistic.HAND_CONNECTIONS,left_draw_spec,left_connect_spec)
            x_list=[lm.x*w for lm in results.left_hand_landmarks.landmark]
            y_list=[lm.y*h for lm in results.left_hand_landmarks.landmark]
            x_min=max(0,int(min(x_list))-10)
            x_max=min(w,int(max(x_list))+10)
            y_min=max(0,int(min(y_list))-10)
            y_max=min(h,int(max(y_list))+10)
            cv2.rectangle(frame,(x_min,y_min),(x_max,y_max),(255,80,180),2)
            cv2.putText(frame,"LEFT HAND",(x_min,y_min-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,80,180),2)

        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(frame,results.right_hand_landmarks,mp_holistic.HAND_CONNECTIONS,right_draw_spec,right_connect_spec)
            x_list=[lm.x*w for lm in results.right_hand_landmarks.landmark]
            y_list=[lm.y*h for lm in results.right_hand_landmarks.landmark]
            x_min=max(0,int(min(x_list))-10)
            x_max=min(w,int(max(x_list))+10)
            y_min=max(0,int(min(y_list))-10)
            y_max=min(h,int(max(y_list))+10)
            cv2.rectangle(frame,(x_min,y_min),(x_max,y_max),(0,220,255),2)
            cv2.putText(frame,"RIGHT HAND",(x_min,y_min-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,220,255),2)

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(frame,results.pose_landmarks,mp_holistic.POSE_CONNECTIONS,pose_draw_spec,pose_connect_spec)
            x_list=[lm.x*w for lm in results.pose_landmarks.landmark if lm.visibility>0.5]
            y_list=[lm.y*h for lm in results.pose_landmarks.landmark if lm.visibility>0.5]

            if x_list and y_list:
                x_min=max(0,int(min(x_list))-10)
                x_max=min(w,int(max(x_list))+10)
                y_min=max(0,int(min(y_list))-10)
                y_max=min(h,int(max(y_list))+10)
                cv2.rectangle(frame,(x_min,y_min),(x_max,y_max),(80,255,120),2)
                cv2.putText(frame,"POSE",(x_min,y_min-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,(80,255,120),2)

        current_time=time.time()
        fps=0.9*fps+0.1/(current_time-prev_time)
        prev_time=current_time

        cv2.putText(frame,f"FPS: {int(fps)}",(20,40),cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)

        cv2.imshow("MediaPipe Holistic",frame)

        if cv2.waitKey(1)&0xFF==ord("q"):
            break

cap.release()
cv2.destroyAllWindows()